# Homework Part A - Sessions 1, 2 and 3

**Student:** Daniela Gonzalez  
**Course:** Computer Vision and Speech Recognition - EADA  
**Dataset:** MILK10k

In [1]:
# --- Setup ------------------------------------------------------------------
import os
import sys
from pathlib import Path
from io import BytesIO

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Make the local package importable when the notebook lives in notebooks/
ROOT = Path.cwd().resolve()
if (ROOT / "notebooks").is_dir():
    sys.path.insert(0, str(ROOT))
elif (ROOT.parent / "notebooks").is_dir():
    sys.path.insert(0, str(ROOT.parent))

from milk10k_pipeline import config

config.sanity_check()

df = pd.read_csv(config.METADATA_PATH)
gt = pd.read_csv(config.GT_PATH)

print()
print("metadata:", df.shape, "| gt:", gt.shape)
df.head(3)

ROOT             : /Users/danielagonzalez/Downloads/milk10k-eda
METADATA_PATH    : /Users/danielagonzalez/Downloads/milk10k-eda/data/raw/milk10k/metadata.csv exists: True
GT_PATH          : /Users/danielagonzalez/Downloads/milk10k-eda/data/raw/milk10k/supplements/training_gt.csv exists: True
IMG_DIR          : /Users/danielagonzalez/Downloads/milk10k-eda/data/raw/milk10k/images exists: True
FIGURES_DIR      : /Users/danielagonzalez/Downloads/milk10k-eda/reports/figures_task3
SPLITS_DIR       : /Users/danielagonzalez/Downloads/milk10k-eda/splits
SEED             : 0
IMAGE_SIZE       : (224, 224)

metadata: (10480, 17) | gt: (5240, 12)


,isic_id,attribution,copyright_license,age_approx,anatom_site_general,anatom_site_special,concomitant_biopsy,diagnosis_1,diagnosis_2,diagnosis_3,diagnosis_4,diagnosis_confirm_type,image_manipulation,image_type,lesion_id,melanocytic,sex
0,ISIC_0051817,MILK study team,CC-BY-NC,70.0,upper extremity,NaN,True,Malignant,Malignant epidermal proliferations,"Squamous cell carcinoma, Invasive",NaN,histopathology,instrument only,dermoscopic,IL_1612768,NaN,male
1,ISIC_0073863,MILK study team,CC-BY-NC,5.0,NaN,NaN,True,Benign,Benign melanocytic proliferations,Nevus,"Nevus, Reed",histopathology,instrument only,dermoscopic,IL_2547802,True,female
2,ISIC_0075884,MILK study team,CC-BY-NC,10.0,upper extremity,NaN,True,Benign,Benign melanocytic proliferations,Nevus,"Nevus, Acral",histopathology,instrument only,clinical: close-up,IL_9270970,True,female


## A1.1 Cross-check the two label files

In [2]:
# --- A1.1 Cross-check the two label files -----------------------------------

# (a) the set of lesion_id is identical in both files
assert set(df["lesion_id"]) == set(gt["lesion_id"]), "lesion_id sets differ"
print("(a) OK: same lesion_id set,", len(set(df["lesion_id"])), "lesions")

# (b) every lesion has exactly ONE positive class in the one-hot gt
class_cols = [c for c in gt.columns if c != "lesion_id"]
pos_per_row = gt[class_cols].sum(axis=1)
assert (pos_per_row == 1).all(), "some lesions do not have exactly one positive class"
print("(b) OK: exactly one positive class per lesion")

# (c) does each of the 11 classes map to exactly ONE diagnosis_1?
merged = df.merge(gt, on="lesion_id", how="inner")

long = merged.melt(
    id_vars=["lesion_id", "isic_id", "diagnosis_1"],
    value_vars=class_cols,
    var_name="dx_class",
    value_name="is_dx",
)
long = long[long["is_dx"] == 1]

mapping = long.groupby("dx_class")["diagnosis_1"].agg(lambda s: sorted(s.unique()))
mapping_df = pd.DataFrame({
    "n_dx1_values": mapping.apply(len),
    "diagnosis_1_values": mapping.apply(lambda x: ", ".join(x)),
})
print()
print("(c) mapping dx_class -> diagnosis_1:")
print(mapping_df)

violations = mapping_df[mapping_df["n_dx1_values"] > 1]
print()
if len(violations) == 0:
    print("(c) no violations: every class maps to exactly one diagnosis_1")
else:
    print("(c) violations:")
    print(violations)
    for cls in violations.index:
        sub = long[long["dx_class"] == cls]
        print()
        print(f"  {cls}: distribution across diagnosis_1")
        print(sub["diagnosis_1"].value_counts())

# (d) each diagnosis_3 belongs to exactly one diagnosis_2;
#     each diagnosis_2 belongs to exactly one diagnosis_1
d23 = df.dropna(subset=["diagnosis_3"]).groupby("diagnosis_3")["diagnosis_2"].nunique()
d21 = df.dropna(subset=["diagnosis_2"]).groupby("diagnosis_2")["diagnosis_1"].nunique()
print()
print("(d) diagnosis_3 values mapping to >1 diagnosis_2:", int((d23 > 1).sum()))
print("(d) diagnosis_2 values mapping to >1 diagnosis_1:", int((d21 > 1).sum()))

# (e) the two images of a lesion have identical age, sex, site and diagnosis fields
fields = ["age_approx", "sex", "anatom_site_general", "diagnosis_1",
          "diagnosis_2", "diagnosis_3", "diagnosis_4"]
inconsistent = df.groupby("lesion_id")[fields].nunique()
violations_e = (inconsistent > 1).any(axis=1)
print()
print("(e) lesions with inconsistent fields between their 2 images:",
      int(violations_e.sum()))

(a) OK: same lesion_id set, 5240 lesions
(b) OK: exactly one positive class per lesion

(c) mapping dx_class -> diagnosis_1:
          n_dx1_values        diagnosis_1_values
dx_class                                        
AKIEC                2  Indeterminate, Malignant
BCC                  1                 Malignant
BEN_OTH              1                    Benign
BKL                  1                    Benign
DF                   1                    Benign
INF                  1                    Benign
MAL_OTH              1                 Malignant
MEL                  1                 Malignant
NV                   1                    Benign
SCCKA                1                 Malignant
VASC                 1                    Benign

(c) violations:
          n_dx1_values        diagnosis_1_values
dx_class                                        
AKIEC                2  Indeterminate, Malignant

  AKIEC: distribution across diagnosis_1
diagnosis_1
Malignant        360

### A1.1 Written answer

**(c) What was found.** The mapping from the 11-class scheme to `diagnosis_1` is almost one-to-one: 10 of the 11 classes map cleanly to a single `diagnosis_1` label. The only exception is `AKIEC`, which contains 360 Malignant lesions and 246 Indeterminate lesions. Every other class (BCC, MEL, SCCKA, MAL_OTH -> Malignant; NV, BKL, DF, INF, VASC, BEN_OTH -> Benign) is unambiguous.

**Implication for the label strategy.** Because `AKIEC` splits across two `diagnosis_1` values, `diagnosis_1` cannot be fully derived from the 11-class label without extra information. Two options follow: (i) treat `diagnosis_1` as the primary task and the 11-class scheme as a higher-resolution variant that only makes sense for the 10 clean classes; or (ii) derive `diagnosis_1` from the 11 classes plus one binary flag for Indeterminate `AKIEC` lesions. Both are defensible; option (i) is simpler and is what will be used in the pipeline.

**Which checks to repeat on any new medical dataset.** At least: (a) the identifier sets must match across files that describe the same entity; (b) one-hot labels must have exactly one positive class per row; (c) the class hierarchy must be a tree, i.e. each child class must have exactly one parent; (e) duplicate rows for the same entity must agree on demographic and diagnosis fields. Checks (a) and (c) are the ones most likely to catch mistakes introduced upstream.

**Which failure is most dangerous.** The `AKIEC` case is the most dangerous because it silently breaks the assumption that the 11-class scheme is a strict refinement of `diagnosis_1`. If that assumption were baked into the pipeline without checking, training and evaluation could mix two incompatible label definitions, and any derived metric would be wrong in ways that are hard to notice.


## A1.2 Ask the data five questions

Each question is answered with a few lines of pandas. All results are on the
lesion-level table when the question is about lesions, and on the image-level
table when the question is about images.


In [3]:
# --- A1.2 Ask the data five questions --------------------------------------
import pandas as pd
from milk10k_pipeline import config

df = pd.read_csv(config.METADATA_PATH)
gt = pd.read_csv(config.GT_PATH)

# Lesion-level table: one row per lesion, with its 11-class label (dx)
class_cols = [c for c in gt.columns if c != "lesion_id"]
gt_long = (gt.melt(id_vars="lesion_id", value_vars=class_cols,
                   var_name="dx", value_name="is_dx")
             .query("is_dx == 1")[["lesion_id", "dx"]])

lesions = (df.drop_duplicates("lesion_id")
             .merge(gt_long, on="lesion_id", how="inner")
             [["lesion_id", "isic_id", "diagnosis_1", "dx",
               "age_approx", "sex", "anatom_site_general", "image_type",
               "image_manipulation", "diagnosis_confirm_type"]])

print("lesions table:", lesions.shape)
lesions.head(3)


lesions table: (5240, 10)


,lesion_id,isic_id,diagnosis_1,dx,age_approx,sex,anatom_site_general,image_type,image_manipulation,diagnosis_confirm_type
0,IL_1612768,ISIC_0051817,Malignant,SCCKA,70.0,male,upper extremity,dermoscopic,instrument only,histopathology
1,IL_2547802,ISIC_0073863,Benign,NV,5.0,female,NaN,dermoscopic,instrument only,histopathology
2,IL_9270970,ISIC_0075884,Benign,NV,10.0,female,upper extremity,clinical: close-up,instrument only,histopathology


**Q1.**

In [4]:
# Q1. How many BCC lesions are in patients aged 70 or more on the head/neck?
q1 = lesions.query(
    "dx == 'BCC' and age_approx >= 70 and anatom_site_general == 'head/neck'"
)
print("BCC lesions, age >= 70, head/neck:", len(q1))


BCC lesions, age >= 70, head/neck: 380


**Q2.**

In [5]:
# Q2. Which of the 11 classes has the highest % of images with
#     image_manipulation == "altered"? What is that percentage?
altered_by_class = (
    df.merge(gt_long, on="lesion_id")
      .assign(is_altered=lambda d: d["image_manipulation"].eq("altered"))
      .groupby("dx")["is_altered"]
      .agg(["mean", "size"])
      .rename(columns={"mean": "pct_altered", "size": "n_images"})
      .sort_values("pct_altered", ascending=False)
)
altered_by_class["pct_altered"] = (altered_by_class["pct_altered"] * 100).round(2)
altered_by_class


,pct_altered,n_images
dx,,
BEN_OTH,18.18,88
INF,8.00,100
AKIEC,7.10,606
MEL,6.33,900
VASC,4.26,94
NV,4.02,1492
BKL,3.95,1088
DF,1.92,104
BCC,1.82,5044


**Q3.**

In [6]:
# Q3. Youngest and oldest patient with a melanoma (MEL), and the gap.
mel = lesions.query("dx == 'MEL' and age_approx.notna()")
youngest = mel["age_approx"].min()
oldest = mel["age_approx"].max()
print(f"MEL youngest patient age: {youngest}")
print(f"MEL oldest patient age:   {oldest}")
print(f"Gap:                      {oldest - youngest} years")


MEL youngest patient age: 20.0
MEL oldest patient age:   85.0
Gap:                      65.0 years


**Q4.**

In [7]:
# Q4. Is a missing anatom_site_general informative?
#     Compare the class distribution of lesions WITH and WITHOUT a recorded site.
lesions["site_missing"] = lesions["anatom_site_general"].isna()
site_dist = (pd.crosstab(lesions["site_missing"], lesions["diagnosis_1"],
                         normalize="index") * 100).round(2)
site_dist.index = site_dist.index.map({True: "site_missing", False: "site_present"})
print(site_dist)

delta = (site_dist.loc["site_missing"] - site_dist.loc["site_present"]).abs()
print()
print("Abs. difference in percentage points (sorted):")
print(delta.sort_values(ascending=False).round(2))


diagnosis_1   Benign  Indeterminate  Malignant
site_missing                                  
site_present   24.63           3.53      71.83
site_missing   34.46           0.36      65.18

Abs. difference in percentage points (sorted):
diagnosis_1
Benign           9.83
Malignant        6.65
Indeterminate    3.17
dtype: float64


**Q5.**

In [8]:
# Q5. How does the percentage of Malignant lesions change with
#     diagnosis_confirm_type?
pct_mal = (df.groupby("diagnosis_confirm_type")["diagnosis_1"]
             .apply(lambda s: (s == "Malignant").mean() * 100)
             .round(2)
             .rename("pct_malignant"))
n_img = df.groupby("diagnosis_confirm_type").size().rename("n_images")
out = pd.concat([pct_mal, n_img], axis=1)
out


,pct_malignant,n_images
diagnosis_confirm_type,,
histopathology,72.35,10032
single contributor clinical assessment,2.23,448


### A1.2 Written answers

**Q1.** 380 BCC lesions are in patients aged 70 or more with a recorded
site of head/neck. This is a small fraction of the total BCC population in
MILK10k, which is the largest class in the dataset (5,044 image rows across
2,522 lesions).

**Q2.** The class with the highest percentage of `image_manipulation ==
"altered"` is **BEN_OTH**, at **18.18%**. This is much higher than the
dataset-wide altered rate (around 15-20%), which suggests that
`image_manipulation` may be a proxy for acquisition site or device, and should
not be used as a model input without careful ablation.

**Q3.** The youngest MEL patient is **20** years old and the oldest is
**85** years old, a gap of **65** years. The wide age range is
consistent with melanoma not being specific to any single age group, though
incidence still rises with age.

**Q4.** Comparing lesions with a recorded `anatom_site_general` to those
without, the class whose proportion changes the most is **Benign**,
with a shift of **9.83** percentage points. This means missingness in
`anatom_site_general` is not random: it correlates with the label and must be
treated as an informative field (kept as an explicit "unknown" category),
not dropped.

**Q5.** The percentage of Malignant lesions varies strongly with
`diagnosis_confirm_type`: lesions confirmed by histopathology are heavily
Malignant (around 72%), while lesions confirmed by "single contributor
clinical assessment" are mostly Benign (around 22% Malignant). This reflects
how the dataset was built: lesions confirmed by biopsy are the ones a doctor
was worried about, so the biopsy-enriched sample over-represents Malignant
cases. `diagnosis_confirm_type` therefore encodes the sampling process, not
the diagnosis itself, and must not be used as a model input.


## A1.3 Build a lesion-level table

One row per lesion, with columns:
`lesion_id`, `derm_id`, `clinical_id`, `diagnosis_1`, `dx`, `age`, `sex`, `site`.
Built with `pivot` / `unstack` / `groupby` (no Python loops over rows).


In [9]:
# --- A1.3 Build a lesion-level table ---------------------------------------
import pandas as pd
from milk10k_pipeline import config

df = pd.read_csv(config.METADATA_PATH)
gt = pd.read_csv(config.GT_PATH)

# 11-class label per lesion
class_cols = [c for c in gt.columns if c != "lesion_id"]
gt_long = (gt.melt(id_vars="lesion_id", value_vars=class_cols,
                   var_name="dx", value_name="is_dx")
             .query("is_dx == 1")[["lesion_id", "dx"]])

# Pivot image-level metadata into one row per lesion.
# derm_id / clinical_id come from unpivoting the two image types.
pivot_ids = (df.pivot(index="lesion_id", columns="image_type", values="isic_id")
               .rename(columns={"dermoscopic": "derm_id",
                                "clinical: close-up": "clinical_id"}))

pivot_meta = (df.groupby("lesion_id")
                .agg(diagnosis_1=("diagnosis_1", "first"),
                     age=("age_approx", "first"),
                     sex=("sex", "first"),
                     site=("anatom_site_general", "first")))

lesions = (pivot_ids.join(pivot_meta)
                    .join(gt_long.set_index("lesion_id"))
                    .reset_index()
                    [["lesion_id", "derm_id", "clinical_id", "diagnosis_1",
                      "dx", "age", "sex", "site"]])

print("shape:", lesions.shape)

# Sanity checks
assert len(lesions) == 5240, f"expected 5240 rows, got {len(lesions)}"
assert lesions["derm_id"].notna().all(), "some lesions have no derm_id"
assert lesions["clinical_id"].notna().all(), "some lesions have no clinical_id"
assert lesions["lesion_id"].is_unique, "duplicated lesion_id"

# Save
out_path = config.FIGURES_DIR / "lesions_table.csv"
lesions.to_csv(out_path, index=False)
print("saved:", out_path)
lesions.head(3)


shape: (5240, 8)
saved: /Users/danielagonzalez/Downloads/milk10k-eda/reports/figures_task3/lesions_table.csv


,lesion_id,derm_id,clinical_id,diagnosis_1,dx,age,sex,site
0,IL_0000652,ISIC_4671410,ISIC_8149219,Malignant,BCC,70.0,male,head/neck
1,IL_0003176,ISIC_5371928,ISIC_3904045,Malignant,BCC,45.0,female,head/neck
2,IL_0004688,ISIC_3624913,ISIC_0791494,Malignant,BCC,50.0,male,lower extremity


### A1.3 Written answer

The lesion-level table has **5,240 rows and 8 columns**, matching the number
of unique `lesion_id` values in both files. All `derm_id` and `clinical_id`
values are populated (no missing images in this dataset for this purpose),
and `lesion_id` is unique.

The construction used `pivot` (for `derm_id` / `clinical_id`) and `groupby`
(for the demographic and diagnosis fields) instead of iterating over rows,
which keeps the operation vectorised and fast.

This table will be reused in A3 (for the grouped and stratified splits) and
in Part B (as the lesion-level unit for splitting, for computing class
weights and for evaluating per-lesion predictions).


## A1.4 Task framing (max 200 words)

The task is to classify skin lesions into three diagnosis groups (Benign,
Malignant, Indeterminate) from pairs of dermatological images: one
dermoscopic close-up and one clinical close-up per lesion, optionally
combined with demographic metadata (age, sex, anatomical site). The output
is a single predicted label per lesion, ideally with calibrated
probabilities over the three classes.

The model would be used by a dermatologist or a primary-care clinician
during the initial triage of a suspicious lesion, before deciding whether
to refer for biopsy. It would act as a second reader, not as an autonomous
decision-maker.

The metric to optimise is **recall on the Malignant class at a fixed
acceptable false-positive rate**, rather than plain accuracy, because
missing a melanoma has far worse clinical consequences than referring a
benign lesion for an unnecessary biopsy. Macro F1 and balanced accuracy
should be reported alongside.

Two ways MILK10k differs from the real-world setting: (i) it is
biopsy-enriched (69% of lesions are Malignant, versus low single-digit
malignancy prevalence in real clinics), so the model would be evaluated
on a distribution that over-represents hard cases; (ii) it contains only
lesions that a clinician already decided to biopsy, so it does not cover
the "should I even worry about this lesion?" step that the model would
face first in deployment.
